# Восстановление пунктуации и регистра (RU) — rubert-tiny2, две головы

**Перед запуском (Settings / Input справа):**
1. **Input → Upload dataset**: папка `punct_ru/ml` (нужны `punct_data.py` и `eval/conversational_ru.txt`).
2. **Input → Add dataset**: `yutkin/corpus-of-russian-news-articles-from-lenta` (Lenta.ru).
3. **Internet: On** (нужен подтверждённый по телефону аккаунт) — для rubert-tiny2 и Википедии.
4. Первый прогон: `DRY_RUN = True`, **Accelerator: None** (CPU, квота GPU не тратится) — проверка всего пайплайна за несколько минут.
5. Боевой прогон: `DRY_RUN = False`, **Accelerator: GPU (P100)**, **Save Version → Save & Run All (Commit)**,
   чтобы обучение шло в фоне. Результат — папка `export/` во вкладке Output.

In [ ]:
from pathlib import Path

DRY_RUN = True  # True: крошечные данные на CPU для проверки пайплайна. Для обучения -> False.

MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_WORDS = 64            # максимум слов в обучающем примере
MAX_TOKENS = 160          # 64 слова ≈ 100 токенов, с запасом
EXAMPLES_PER_SOURCE = 2_000 if DRY_RUN else 500_000
WIKI_ARTICLES = 300 if DRY_RUN else 200_000
EPOCHS = 1 if DRY_RUN else 2
BATCH_SIZE = 64
LR = 1e-4
SEED = 42
WORK = Path("/kaggle/working")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)

In [ ]:
import glob, json, random, time
from collections import Counter

import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report

code_dir = Path(glob.glob("/kaggle/input/**/punct_data.py", recursive=True)[0]).parent
sys.path.insert(0, str(code_dir))
import punct_data as pdata
from punct_data import IGNORE, PUNCT_LABELS, CASE_LABELS

eval_matches = glob.glob("/kaggle/input/**/conversational_ru.txt", recursive=True)
if eval_matches:
    EVAL_FILE = Path(eval_matches[0])
else:
    EVAL_FILE = WORK / "conversational_ru.txt"
    EVAL_FILE.write_text("""Привет, ты сегодня будешь на созвоне?
Слушай, я посмотрел твой отчёт и в целом всё хорошо, но в третьем разделе цифры не сходятся.
Короче, я думаю, что нам стоит перенести встречу на четверг.
Если честно, я не уверен, что мы успеем к пятнице.
Спасибо большое! Всё получил, завтра посмотрю.
Мы с Сашей вчера обсуждали проект и решили, что начнём с дизайна.
Можешь скинуть ссылку на документ? Я его где-то потерял.
Когда будет готова новая версия, напиши мне, пожалуйста.
Я позвонил в банк, но там сказали, что карту выпустят только через неделю.
Добрый день! Направляю вам договор на согласование.
Ну не знаю, мне кажется, это слишком дорого.
Вчера ходили в кино, фильм так себе, но попкорн был отличный.
Ты где? Мы уже ждём тебя у входа.
В общем, сервер упал ночью, и я до утра его поднимал.
Дима сказал, что пришлёт отчёт в понедельник утром.
Если что, звони, я весь день дома.
Напоминаю, что оплата должна поступить до конца месяца.
Я купил новый ноутбук, и он работает намного быстрее старого.
Мама просила передать, что приедет в субботу вечером.
Как думаешь, стоит ли брать машину в кредит?
Не забудь купить хлеб и молоко по дороге домой.
Офис в Москве переезжает, поэтому на следующей неделе работаем из дома.
Привет! Как дела? Давно не виделись.
Пока ты спал, я успел приготовить завтрак и погулять с собакой.
Компания Яндекс выпустила обновление, которое исправляет эту ошибку.
Честно говоря, я ожидал от этой встречи большего.
Отправил тебе файлы на почту, проверь, пожалуйста, всё ли открывается.
Если бы я знал заранее, то обязательно пришёл бы.
Ребята, кто сегодня идёт обедать? Я угощаю!
Наш партнёр из США подтвердил заказ, и отгрузка будет в понедельник.""", encoding="utf-8")

random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("code:", code_dir, "| device:", device)

## 1. Датасет: OpenSubtitles (разговорный) + Lenta.ru (новости) + Википедия

In [ ]:
from datasets import load_dataset

t0 = time.time()

# 1.1 OpenSubtitles: живые диалоги, вопросы ?, восклицания !
subs_gz = WORK / "data/raw/subs.txt.gz"
if not subs_gz.exists():
    print("Скачиваем русские субтитры OpenSubtitles...")
    subs_gz.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["wget", "-q", "-O", str(subs_gz),
                    "https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/mono/ru.txt.gz"], check=True)
print("subs:", subs_gz)
print(pdata.build_dataset([subs_gz], WORK / "data/subs", max_words=MAX_WORDS,
                          max_examples=EXAMPLES_PER_SOURCE, seed=SEED, is_subtitles=True))

# 1.2 Lenta.ru: новости, сложные предложения, запятые перед что/который
lenta_matches = glob.glob("/kaggle/input/**/lenta*.csv", recursive=True)
if lenta_matches:
    lenta_csv = lenta_matches[0]
else:
    print("Датасет Lenta не найден в /kaggle/input, скачиваем и распаковываем...")
    lenta_dir = WORK / "data/raw"
    lenta_dir.mkdir(parents=True, exist_ok=True)
    lenta_gz = lenta_dir / "lenta-ru-news.csv.gz"
    lenta_csv = str(lenta_dir / "lenta-ru-news.csv")
    if not Path(lenta_csv).exists():
        if not lenta_gz.exists():
            subprocess.run(["wget", "-q", "-O", str(lenta_gz),
                            "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"], check=True)
        subprocess.run(["gunzip", "-f", str(lenta_gz)], check=True)
print("lenta:", lenta_csv)
print(pdata.build_dataset([lenta_csv], WORK / "data/lenta", max_words=MAX_WORDS,
                          max_examples=EXAMPLES_PER_SOURCE, seed=SEED))

# 1.3 Википедия: энциклопедический стиль, факты, заглавные буквы
wiki_raw = WORK / "data/wiki_raw.jsonl"
wiki_raw.parent.mkdir(parents=True, exist_ok=True)
with wiki_raw.open("w", encoding="utf-8") as f:
    for i, art in enumerate(load_dataset("wikimedia/wikipedia", "20231101.ru", split="train", streaming=True)):
        if i >= WIKI_ARTICLES:
            break
        f.write(json.dumps({"text": art["text"]}, ensure_ascii=False) + "\n")
print(pdata.build_dataset([wiki_raw], WORK / "data/wiki", max_words=MAX_WORDS,
                          max_examples=EXAMPLES_PER_SOURCE, seed=SEED))
wiki_raw.unlink()
print(f"data ready in {time.time() - t0:.0f}s")

In [ ]:
def read_split(name):
    rows = []
    for src in ("subs", "lenta", "wiki"):
        p = WORK / "data" / src / f"{name}.jsonl"
        if p.exists():
            with p.open(encoding="utf-8") as f:
                rows += [json.loads(line) for line in f]
    random.shuffle(rows)
    return rows

train_rows, val_rows, test_rows = read_split("train"), read_split("val"), read_split("test")
print(f"Всего примеров -> train: {len(train_rows)}, val: {len(val_rows)}, test: {len(test_rows)}")

## 2. Модель: общий энкодер + головы пунктуации и регистра

In [ ]:
class PunctModel(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.punct_head = nn.Linear(hidden, len(PUNCT_LABELS))
        self.case_head = nn.Linear(hidden, len(CASE_LABELS))

    def forward(self, input_ids, attention_mask):
        x = self.dropout(self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state)
        return self.punct_head(x), self.case_head(x)


tok = AutoTokenizer.from_pretrained(MODEL_NAME)


def collate(rows):
    enc = tok([r["words"] for r in rows], is_split_into_words=True, truncation=True,
              max_length=MAX_TOKENS, padding=True, return_tensors="pt")
    punct = [pdata.align_labels(enc.word_ids(i), r["punct"]) for i, r in enumerate(rows)]
    case = [pdata.align_labels(enc.word_ids(i), r["case"]) for i, r in enumerate(rows)]
    return enc["input_ids"], enc["attention_mask"], torch.tensor(punct), torch.tensor(case)


def class_weights(rows, key, n):
    """Inverse-sqrt frequency: 'O' dominates (~85%), без весов модель ленится ставить знаки."""
    cnt = Counter(label for r in rows for label in r[key])
    freq = torch.tensor([cnt.get(i, 0) + 1 for i in range(n)], dtype=torch.float)
    w = (freq.sum() / freq) ** 0.5
    return w / w.mean()


w_punct = class_weights(train_rows, "punct", len(PUNCT_LABELS))
w_case = class_weights(train_rows, "case", len(CASE_LABELS))
print("weights punct:", w_punct.tolist(), "case:", w_case.tolist())

In [ ]:
@torch.no_grad()
def evaluate(rows, title):
    model.eval()
    pp, pt, cp, ct = [], [], [], []
    for ids, mask, p, c in DataLoader(rows, batch_size=256, collate_fn=collate):
        lp, lc = model(ids.to(device), mask.to(device))
        m = p != IGNORE
        pp += lp.argmax(-1).cpu()[m].tolist(); pt += p[m].tolist()
        cp += lc.argmax(-1).cpu()[m].tolist(); ct += c[m].tolist()
    print(f"== {title}: punctuation")
    print(classification_report(pt, pp, labels=range(len(PUNCT_LABELS)), target_names=PUNCT_LABELS, digits=3, zero_division=0))
    print(f"== {title}: case")
    print(classification_report(ct, cp, labels=range(len(CASE_LABELS)), target_names=CASE_LABELS, digits=3, zero_division=0))


@torch.no_grad()
def restore(text):
    """Сырой текст -> текст с пунктуацией. Слова пользователя не меняются, только знаки и регистр."""
    model.eval()
    words = pdata.WORD_RE.findall(text)
    if not words:
        return text
    enc = tok([pdata.normalize_word(w) for w in words], is_split_into_words=True,
              truncation=True, max_length=512, return_tensors="pt")
    lp, lc = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
    punct, case, prev = [0] * len(words), [0] * len(words), None
    for t, wid in enumerate(enc.word_ids()):
        if wid is not None and wid != prev:
            punct[wid], case[wid] = lp[0, t].argmax().item(), lc[0, t].argmax().item()
        prev = wid
    return pdata.apply_labels(words, punct, case)


def conversational_eval():
    lines = EVAL_FILE.read_text(encoding="utf-8").splitlines()
    exact = 0
    for line in lines:
        out = restore(" ".join(w.lower() for w in pdata.WORD_RE.findall(line)))
        exact += out == line
        if out != line:
            print(f"  ожидалось: {line}\n  модель:    {out}\n")
    print(f"== разговорный набор: {exact}/{len(lines)} фраз полностью верно")

## 3. Обучение

In [ ]:
model = PunctModel(MODEL_NAME).to(device)
loader = DataLoader(train_rows, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=2)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(loader)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * total_steps), total_steps)
use_amp = device == "cuda"
scaler = torch.amp.GradScaler(enabled=use_amp)
loss_punct = nn.CrossEntropyLoss(weight=w_punct.to(device), ignore_index=IGNORE)
loss_case = nn.CrossEntropyLoss(weight=w_case.to(device), ignore_index=IGNORE)

for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    for step, (ids, mask, p, c) in enumerate(loader):
        ids, mask, p, c = ids.to(device), mask.to(device), p.to(device), c.to(device)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            lp, lc = model(ids, mask)
            loss = loss_punct(lp.flatten(0, 1), p.flatten()) + loss_case(lc.flatten(0, 1), c.flatten())
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        if step % 500 == 0:
            print(f"epoch {epoch} step {step}/{len(loader)} loss {loss.item():.4f} {time.time() - t0:.0f}s")
    torch.save(model.state_dict(), WORK / "model.pt")
    evaluate(val_rows, f"val, epoch {epoch}")
    conversational_eval()

## 4. Итоговая оценка на test

In [ ]:
evaluate(test_rows, "test")
conversational_eval()
for s in ["привет саша я вчера звонил но ты не ответил перезвони когда сможешь",
          "короче я думаю что нам стоит перенести встречу ты как"]:
    print(restore(s))

## 5. Экспорт в ONNX (int8) + проверка совпадения с PyTorch

In [ ]:
import numpy as np
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

export_dir = WORK / "export"
export_dir.mkdir(exist_ok=True)
fp32, int8 = WORK / "model.fp32.onnx", export_dir / "model.int8.onnx"

model.eval().cpu()
device = "cpu"
dummy = tok(["привет", "как", "дела"], is_split_into_words=True, return_tensors="pt")
names = ["input_ids", "attention_mask", "punct_logits", "case_logits"]
kwargs = dict(input_names=names[:2], output_names=names[2:], opset_version=17,
              dynamic_axes={n: {0: "batch", 1: "seq"} for n in names})
try:
    torch.onnx.export(model, (dummy["input_ids"], dummy["attention_mask"]), str(fp32), dynamo=False, **kwargs)
except TypeError:  # older torch without the `dynamo` argument
    torch.onnx.export(model, (dummy["input_ids"], dummy["attention_mask"]), str(fp32), **kwargs)
quantize_dynamic(str(fp32), str(int8), weight_type=QuantType.QInt8)

tok.save_pretrained(export_dir)  # vocab.txt и конфиг токенизатора для C#
(export_dir / "labels.json").write_text(json.dumps({
    "punct": PUNCT_LABELS, "punct_chars": pdata.PUNCT_CHARS, "case": CASE_LABELS,
    "word_regex": pdata.WORD_RE.pattern, "normalize": "lowercase, ё -> е", "max_tokens": 512,
}, ensure_ascii=False, indent=2), encoding="utf-8")

sess = ort.InferenceSession(str(int8), providers=["CPUExecutionProvider"])
agree = total = 0
with torch.no_grad():
    for ids, mask, p, c in DataLoader(val_rows[:2000], batch_size=64, collate_fn=collate):
        tp, _ = model(ids, mask)
        op, _ = sess.run(None, {"input_ids": ids.numpy(), "attention_mask": mask.numpy()})
        m = (p != IGNORE).numpy()
        agree += int((tp.argmax(-1).numpy()[m] == op.argmax(-1)[m]).sum()); total += int(m.sum())
print(f"int8 vs PyTorch: совпадение меток пунктуации {agree / total:.2%}")

enc = tok(["слово"] * 60, is_split_into_words=True, return_tensors="np")
t0 = time.perf_counter()
for _ in range(20):
    sess.run(None, {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]})
print(f"абзац 60 слов на CPU: {(time.perf_counter() - t0) / 20 * 1000:.1f} мс")
print({f.name: f"{f.stat().st_size / 1e6:.1f} MB" for f in export_dir.iterdir()})